# OffScript — Phase 1: Data Collection

This notebook retrieves pitch-by-pitch Statcast data for a curated roster of
15 MLB pitchers across the 2023 and 2024 seasons using the `pybaseball` library.

**Goals:**
- Pull raw Statcast data for each pitcher via MLBAM player ID
- Trim each dataset to the columns relevant for pitch selection modeling
- Engineer two context features: count state and score differential
- Combine all pitchers into a single dataset and persist to parquet

**Output:** `data/processed/pitcher_data.parquet`

## 1. Setup

In [1]:
import pandas as pd
from pybaseball import statcast_pitcher, cache

# Prevent re-downloading Statcast data on every run
cache.enable()

print("Imports successful")

Imports successful


## 2. Pitcher Roster

The roster covers four pitcher archetypes chosen to ensure the deviation
analysis in Phase 3 captures a range of pitch selection styles:

| Archetype | Pitchers |
|---|---|
| Power Arms | Cole, Strider, Burnes |
| Finesse | Wheeler, Hendricks, Sale |
| Ground Ball | Webb, Valdez, Stroman |
| Veterans | Scherzer, Verlander |
| Deviation Candidates | Kikuchi, Cease, Ryan, Cortes |

MLBAM IDs were confirmed via `pybaseball.playerid_lookup` in notebook 01.

In [2]:
# Pitcher name → MLBAM ID mapping
PITCHERS = {
    # Power Arms
    "Gerrit Cole":      543037,
    "Spencer Strider":  675911,
    "Corbin Burnes":    669203,
    # Finesse
    "Zack Wheeler":     554430,
    "Kyle Hendricks":   543294,
    "Chris Sale":       519242,
    # Ground Ball
    "Logan Webb":       657277,
    "Framber Valdez":   664285,
    "Marcus Stroman":   543135,
    # Veterans
    "Max Scherzer":     453286,
    "Justin Verlander": 434378,
    # Deviation Candidates
    "Yusei Kikuchi":    579328,
    "Dylan Cease":      656302,
    "Joe Ryan":         664970,
    "Nestor Cortes":    641482,
}

START_DATE = "2023-03-30"
END_DATE = "2024-11-01"

print(f"Roster size: {len(PITCHERS)} pitchers")
print(f"Date range:  {START_DATE} to {END_DATE}")

Roster size: 15 pitchers
Date range:  2023-03-30 to 2024-11-01


## 3. Column Selection

Statcast returns approximately 90 columns per pitch. We retain only the subset
needed for pitch selection modeling, covering:

- **Identity:** game date, pitcher ID, player name
- **Pitch characteristics:** type, name, velocity, movement (pfx), location (plate)
- **Game context:** count, baserunners, batter handedness, inning, score
- **Outcome:** event, description

In [3]:
COLS_OF_INTEREST = [
    "game_date",
    "pitcher",
    "player_name",
    "pitch_type",
    "pitch_name",
    "release_speed",
    "pfx_x", "pfx_z",
    "plate_x", "plate_z",
    "balls", "strikes",
    "on_1b", "on_2b", "on_3b",
    "stand", "p_throws",
    "events", "description",
    "inning", "home_score", "away_score",
    "batter",
    "bat_score",
    "fld_score",
]

print(f"Retaining {len(COLS_OF_INTEREST)} of ~90 Statcast columns")

Retaining 25 of ~90 Statcast columns


## 4. Data Collection

Each pitcher is fetched individually via `statcast_pitcher`. Two features are
engineered immediately on retrieval so they are available in the combined
dataset from the start:

- **`count`** — ball-strike state as a string label (e.g. `'1-2'`)
- **`score_diff`** — home minus away score at the time of each pitch

Failures are caught and logged per pitcher so a single bad API response does
not abort the full collection run.

In [4]:
all_data = []
failed = []

for name, pid in PITCHERS.items():
    print(f"Pulling data for {name}...")
    try:
        df = statcast_pitcher(START_DATE, END_DATE, pid)
        df = df[COLS_OF_INTEREST].copy()
        df["count"] = df["balls"].astype(str) + "-" + df["strikes"].astype(str)
        df["score_diff"] = df["home_score"] - df["away_score"]
        df["pitcher_name"] = name
        all_data.append(df)
        print(f"  OK — {len(df):,} pitches")
    except Exception as e:
        print(f"  FAILED — {e}")
        failed.append(name)

if failed:
    print(f"\nFailed to retrieve data for: {failed}")

Pulling data for Gerrit Cole...
  OK — 5,318 pitches
Pulling data for Spencer Strider...
Gathering Player Data
  OK — 3,657 pitches
Pulling data for Corbin Burnes...
Gathering Player Data
  OK — 6,366 pitches
Pulling data for Zack Wheeler...
Gathering Player Data
  OK — 7,010 pitches
Pulling data for Kyle Hendricks...
Gathering Player Data
  OK — 4,525 pitches
Pulling data for Chris Sale...
Gathering Player Data
  OK — 4,628 pitches
Pulling data for Logan Webb...
Gathering Player Data
  OK — 6,590 pitches
Pulling data for Framber Valdez...
Gathering Player Data
  OK — 6,014 pitches
Pulling data for Marcus Stroman...
Gathering Player Data
  OK — 5,521 pitches
Pulling data for Max Scherzer...
Gathering Player Data
  OK — 3,283 pitches
Pulling data for Justin Verlander...
Gathering Player Data
  OK — 4,461 pitches
Pulling data for Yusei Kikuchi...
Gathering Player Data
  OK — 5,938 pitches
Pulling data for Dylan Cease...
  OK — 6,722 pitches
Pulling data for Joe Ryan...
Gathering Player D

## 5. Combine and Save

In [5]:
combined = pd.concat(all_data, ignore_index=True)

print(f"Total pitches: {len(combined):,}")
print(f"Pitchers collected: {combined['pitcher_name'].nunique()}")
print("\nPitch count by pitcher:")
print(combined["pitcher_name"].value_counts())

Total pitches: 74,170
Pitchers collected: 14

Pitch count by pitcher:
pitcher_name
Zack Wheeler        7010
Dylan Cease         6722
Logan Webb          6590
Corbin Burnes       6366
Framber Valdez      6014
Yusei Kikuchi       5938
Marcus Stroman      5521
Gerrit Cole         5318
Chris Sale          4628
Kyle Hendricks      4525
Justin Verlander    4461
Nestor Cortes       4137
Spencer Strider     3657
Max Scherzer        3283
Name: count, dtype: int64


In [6]:
OUTPUT_PATH = "../data/processed/pitcher_data.parquet"

combined.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {combined.shape}")

Saved: ../data/processed/pitcher_data.parquet
Shape: (74170, 28)
